# 🏠 Boston House Price Prediction
**Using the uploaded `HousingData__1_.csv` dataset**

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print('✅ Libraries loaded')

## 1. Load Dataset
> ✅ Using the uploaded `HousingData__1_.csv` (the original code used `data/boston_housing.csv` which is a **different, synthetic file**).

In [ ]:
# ── Load the uploaded HousingData__1_.csv ─────────────────
# NOTE: Update this path to wherever you have placed HousingData_.csv
df = pd.read_csv('HousingData_.csv')
print(f'Shape: {df.shape}')
df.head()

: 

## 2. Exploratory Data Analysis

In [ ]:
df.describe().round(2)

In [ ]:
# ── Missing values ────────────────────────────────────────
# HousingData__1_.csv has missing values in several columns — handle them!
print('Missing values before imputation:')
print(df.isnull().sum())

# Impute with column median (robust to outliers)
for col in df.columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

print('\nMissing values after imputation:')
print(df.isnull().sum())

In [ ]:
# ── Distribution of target variable ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['MEDV'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of House Prices (MEDV)')
axes[0].set_xlabel('Price ($1000s)')

axes[1].boxplot(df['MEDV'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Boxplot of House Prices')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Key relationships ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(df['RM'], df['MEDV'], alpha=0.5, color='steelblue', s=20)
axes[0].set_xlabel('Avg Rooms (RM)')
axes[0].set_ylabel('Price (MEDV)')
axes[0].set_title('Rooms vs Price')

axes[1].scatter(df['LSTAT'], df['MEDV'], alpha=0.5, color='coral', s=20)
axes[1].set_xlabel('% Lower Status (LSTAT)')
axes[1].set_ylabel('Price (MEDV)')
axes[1].set_title('LSTAT vs Price')
plt.tight_layout()
plt.show()

## 3. Preprocessing & Train/Test Split

In [ ]:
X = df.drop('MEDV', axis=1)
y = df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')

## 4. Model Training & Evaluation

In [ ]:
models = {
    'Linear Regression':  LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Lasso Regression':   Lasso(alpha=0.1),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    cv = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2')
    results.append({
        'Model':   name,
        'RMSE':    round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAE':     round(mean_absolute_error(y_test, y_pred), 4),
        'R²':      round(r2_score(y_test, y_pred), 4),
        'CV R²':   round(cv.mean(), 4),
    })

df_res = pd.DataFrame(results)
df_res.sort_values('R²', ascending=False).reset_index(drop=True)

## 5. Best Model — Actual vs Predicted & Residuals

In [ ]:
best_model = models['Gradient Boosting']
y_pred = best_model.predict(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', s=30)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual ($000s)')
axes[0].set_ylabel('Predicted ($000s)')
axes[0].set_title('Actual vs Predicted — Gradient Boosting')
axes[0].legend()

residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='coral', s=30)
axes[1].axhline(0, color='navy', lw=2, linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
rf = models['Random Forest']
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_df))[::-1])
plt.bar(feat_df['Feature'], feat_df['Importance'], color=colors, edgecolor='white')
plt.xticks(rotation=45, ha='right')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(feat_df.head())

## 7. Save Best Model

In [ ]:
import os
os.makedirs('models', exist_ok=True)
bundle = {'model': best_model, 'scaler': scaler, 'feature_names': X.columns.tolist()}
joblib.dump(bundle, 'models/best_model.pkl')
print('Model saved to models/best_model.pkl')